# KHILONA: Modern CNN Deep Learning Model

This notebook implements a Convolutional Neural Network (CNN) using Keras for classifying toy parts by color (blue, yellow, purple) and assigning them to conveyor belts (A, B, C). 

### **Approach:**
- **Data Loading**: Using `tf.keras.utils.image_dataset_from_directory` for a clean 80/20 split from each color folder.
- **Architecture**: 2D Convolutional layers with MaxPooling and Dense layers.
- **Output**: Color prediction + Belt mapping (A, B, C).

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, utils
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

In [ ]:
# Configuration
dataset_path = "src/data"
img_size = (64, 64)
batch_size = 32
epochs = 15
classes = ["blue", "yellow", "purple"]
belts = ["A", "B", "C"]

In [ ]:
# Data Loading and Splitting (80% Train, 20% Validation)
# Using image_dataset_from_directory for efficient, modern data loading
train_ds = utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='categorical',
    class_names=classes
)

val_ds = utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='categorical',
    class_names=classes
)

# Performance Optimization (Prefetching and Normalization)
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
# CNN Model Architecture
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Flatten(),
    
    layers.Dense(128, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Model Training
history = model.fit(
    train_ds,
    epochs=epochs,
    validation_data=val_ds
)

In [ ]:
# Print Final Scores
train_loss, train_acc = model.evaluate(train_ds, verbose=0)
val_loss, val_acc = model.evaluate(val_ds, verbose=0)

print(f"\n{'='*30}")
print(f"FINAL TRAINING ACCURACY: {train_acc:.4f}")
print(f"FINAL TESTING ACCURACY: {val_acc:.4f}")
print(f"{'='*30}")

In [ ]:
def predict_object(img_path):
    # Load and preprocess image
    img = utils.load_img(img_path, target_size=img_size)
    img_array = utils.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Get prediction
    prediction = model.predict(img_array, verbose=0)
    pred_idx = np.argmax(prediction)
    pred_class = classes[pred_idx]
    belt = belts[pred_idx]

    # Print results
    print(f"Predicted Color: {pred_class.upper()}")
    print(f"Assigned Conveyor Belt: {belt}")
    print(f"Confidence: {prediction[0][pred_idx]:.4f}")
    
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# To test, uncomment below:
# predict_object('path_to_test_image.jpg')

In [ ]:
import cv2

def run_live_camera():
    # Open camera
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Could not open camera.")
        return

    print("Starting Live Detection... Press 'q' to quit.")
    
    while True:
        ret, frame = cap.read()
        if not ret: break

        # Preprocess frame for model
        # Convert BGR (OpenCV) to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Resize to model input size
        resized_frame = cv2.resize(rgb_frame, img_size)
        # Normalize and add batch dimension
        input_array = np.expand_dims(resized_frame / 255.0, axis=0)

        # Get prediction
        prediction = model.predict(input_array, verbose=0)
        pred_idx = np.argmax(prediction)
        pred_class = classes[pred_idx]
        belt = belts[pred_idx]
        confidence = prediction[0][pred_idx]

        # Display info on frame
        label = f"{pred_class.upper()} | Belt: {belt} | {confidence:.2f}"
        cv2.putText(frame, label, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        
        # Show the frame
        cv2.imshow('KHILONA Live Detection', frame)

        # Break on 'q' key
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# To start live camera detection, uncomment below:
# run_live_camera()